<a href="https://colab.research.google.com/github/arunkumar-one/myHomePage/blob/main/Travel_Booking_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -U langchain langchain-core langchain-openai

In [1]:
import os
os.environ["OPENROUTER_API_KEY"] = "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"

In [ ]:
# ==============================
# Travel Booking AI Agent
# LangChain (2025 Modern API)
# OpenRouter Compatible
# ==============================

import os
from typing import Dict

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage


# ------------------------------
# LLM via OpenRouter
# ------------------------------
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.3,
)


# ------------------------------
# BOOKING TOOLS
# ------------------------------

class FlightBookingInput(BaseModel):
    from_city: str = Field(description="City where journey starts")
    to_city: str = Field(description="Destination city")
    flight_time: str = Field(description="Flight borading time")
    flight_company: str = Field(description="Flight company name")

@tool(args_schema=FlightBookingInput)
def book_flight(from_city: str, to_city: str,
                flight_time: str, flight_company: str) -> str:
    """Book a flight ticket."""
    return f"✅ Flight booked from {from_city} to {to_city} at {flight_time} via {flight_company}"


class TrainBookingInput(BaseModel):
    from_city: str = Field(description="City where journey starts")
    to_city: str = Field(description="Destination city")
    train_number: str = Field(description="Train number to book")

@tool(args_schema=TrainBookingInput)
def book_train(from_city: str, to_city: str,
               train_number: str) -> str:
    """Book a train ticket."""
    return f"✅ Train {train_number} booked from {from_city} to {to_city}"

class BusBookingInput(BaseModel):
    from_city: str = Field(description="City where journey starts")
    to_city: str = Field(description="Destination city")
    bus_route_or_id: str = Field(description="Bus number or route for identification")

@tool(args_schema=BusBookingInput)
def book_bus(from_city: str, to_city: str,
             bus_route_or_id: str) -> str:
    """Book a bus ticket."""
    return f"✅ Bus booked from {from_city} to {to_city} via route {bus_route_or_id}"


TOOLS = {
    "book_flight": book_flight,
    "book_train": book_train,
    "book_bus": book_bus,
}


# ------------------------------
# SYSTEM PROMPT
# ------------------------------
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a Travel Booking Assistant for Solo Traveller.

Rules:

1. Talk with the user and suggest travel options.
2. Ask confirmation BEFORE booking.
3. Only perform booking if user clearly confirms.
4. Single User by default
5. Time : default asap after 1 hour
6. Dont ask Train Number , Find the Transport details on your own.
7. Dont prompt for mode of Transport, money is not an issue. But the over all travel cost should not exceed 30 K
8. If travel takes less than a 5 hour, choose bus
9. If travel takes less than 10 hours, choose train
10. If bus travel or Train travel travel takes more than 12 hours, choose Flight.
11. After confirmation, respond with a booking action
   in the format:

ACTION: tool_name | arg1=value1,arg2=value2


Available tools:
- book_flight(from_city: str, to_city: str, flight_time: str, flight_company: str)
- book_train(from_city: str, to_city: str, train_number: str)
- book_bus(from_city: str, to_city: str, bus_route_or_id: str)
"""
    ),
    ("human", "{input}")
])


# ------------------------------
# CREATE CHAIN
# ------------------------------
chain = prompt | llm


# ------------------------------
# MEMORY (Modern LangChain)
# ------------------------------
store: Dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
)

SESSION_ID = "user1"

# ------------------------------
# TOOL EXECUTION PARSER
# ------------------------------
def maybe_execute_tool(response_text: str):
    if "ACTION:" not in response_text:
        return response_text

    try:
        action_line = response_text.split("ACTION:")[1].strip()
        tool_name, args_str = action_line.split("|")

        tool_name = tool_name.strip()
        args = {}

        for pair in args_str.split(","):
          pair = pair.strip()

          # skip invalid parts
          if "=" not in pair:
              continue

          k, v = pair.split("=", 1)   # split only once
          args[k.strip()] = v.strip()

        tool = TOOLS.get(tool_name)

        if tool:
            result = tool.invoke(args)

            history = get_session_history(SESSION_ID)
            history.add_message(
                AIMessage(
                  content=f"""'
                  BOOKING COMPLETED SUCCESSFULLY.
                  Result:{result}

                  Do NOT ask confirmation again.
                  Start fresh conversation for next request.
                  """))

            return result

    except Exception as e:
        return f"⚠️ Tool execution failed: {e}"

    return response_text


# ------------------------------
# CHAT LOOP
# ------------------------------

print("✈️ Travel Agent Ready (type 'exit' to quit)\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    result = chain_with_memory.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": SESSION_ID}},
    )

    output_text = result.content

    # Execute tool if requested
    final_output = maybe_execute_tool(output_text)

    print("\nAgent:", final_output, "\n")


✈️ Travel Agent Ready (type 'exit' to quit)

You: Boo Nagercoil to Trichy

Agent: The travel distance from Nagercoil to Trichy is approximately 6 hours by train. I will book a train for you.

Would you like me to proceed with the booking? 

You: Yes

Agent: ✅ Train 12345 booked from Nagercoil to Trichy 



In [ ]:
!pip install -U langchain langchain-core langchain-openai langchain-community